# Train whole-resistor segmentation in Google Colab

Use a GPU runtime and choose **Runtime → Run all**. This trains a binary resistor/background model with torchvision **LR-ASPP + MobileNetV3-Large**. It does not use Ultralytics.

The notebook now bootstraps through a fresh `git clone` of `main` instead of `raw.githubusercontent.com`, avoiding stale CDN-cached trainer scripts. At startup it prints an explicit `Segmentation Colab version:` line so you can verify the current code is running.

The dataset is downloaded automatically from `v4/resistor_sam3_merged.zip` and its SHA-256 is verified. The loader uses the exact binary `masks_semantic/*.png` produced by SAM 3.1, not bounding-box approximations or the large COCO RLE metadata. All persistent outputs go to `MyDrive/resistor_model/segmentation-v4-lraspp`, separate from m2.

Training prints every batch with epoch, batch %, loss, learning rate and elapsed time. Validation/test output includes Dice and IoU. The launcher captures the trainer's **stdout and stderr** and forwards both through the notebook's Python output stream, so PyTorch startup, epoch progress, Python tracebacks and CUDA errors are visible. If the trainer goes silent, the parent prints heartbeat/process diagnostics every 5 seconds.

If `last.pt` exists in Drive, it is copied first to local `/content` with visible copy progress and training resumes from that local copy. `last.pt` is saved after every epoch, `best.pt` whenever validation Dice improves, and `checkpoints/epoch_NNN.pt` every 5 epochs plus the final epoch. Optimizer, scheduler and AMP scaler state are included.

Default training: 384×384, 60 epochs, batch size 16, weighted cross-entropy + foreground Dice loss. The MobileNetV3 backbone trains from scratch by default, avoiding torchvision's ImageNet-pretrained weights and their potentially dataset-derived terms. Set `USE_PRETRAINED_BACKBONE = True` in the downloaded script only after reviewing those terms for your use case.

After training, the best checkpoint is exported to ONNX and the complete persistent run directory is packaged as `resistor-segmentation-v4-lraspp.zip` for download. If browser download fails, everything remains in Drive.

The exported ONNX returns a single `[N,1,H,W]` resistor probability map for mobile post-processing: threshold → connected component → long-axis/oriented crop → ResistorBandNet.
The **post-training image test** is a separate cell at the end: upload your own photos and inspect predictions without rerunning the training cell.


In [ ]:
import pathlib, shutil, subprocess
bootstrap = pathlib.Path('/content/resistor_model_bootstrap')
if bootstrap.exists():
    shutil.rmtree(bootstrap)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/resistor_model.git', str(bootstrap)], check=True)
subprocess.run(['git', '-C', str(bootstrap), 'rev-parse', '--short', 'HEAD'], check=True)
script = bootstrap / 'colab' / 'train_resistor_segmentation_colab.py'
source = script.read_text(encoding='utf-8')
if 'SEGMENTATION_COLAB_VERSION' not in source or 'PROGRESS_EVERY = 1' not in source or 'run_with_heartbeat' not in source:
    raise RuntimeError('Fresh clone does not contain the expected current Colab trainer. Stop instead of running stale code.')
exec(compile(source, str(script), 'exec'), {'__name__': '__main__'})


## Test the trained segmentation model on your photos

**Run this cell after training** (or independently once `best.pt` exists in Google Drive). It uses the best validation checkpoint, not the latest training epoch. You can upload **multiple JPG/PNG images** and rerun this test cell without retraining. The panel includes segmentation probability, mask, overlay, estimated axis, aligned resistor and horizontal BandNet crop.

If training is still running in another session, avoid exporting while `best.pt` is being written. No input photos are included in training or used to modify model weights.


In [ ]:
# POST-TRAINING IMAGE TEST — rerunnable independently; does not start training.
from pathlib import Path
import runpy
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
repo = Path("/content/resistor_segmentation/resistor_model")
if not (repo / 'colab' / "test_resistor_segmentation_alignment_colab.py").is_file():
    # A separate/reconnected Colab runtime may not have the original clone.
    repo.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/Persie0/resistor_model.git', str(repo)],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-e', str(repo) + '[segmentation,export]',
     'onnxruntime', 'opencv-python-headless', 'matplotlib'],
    check=True,
)
sys.path.insert(0, str(repo / 'src'))
run_dir = Path('/content/drive/MyDrive/resistor_model/segmentation-v4-lraspp')
best = run_dir / 'best.pt'
onnx = run_dir / 'resistor_segmenter_lraspp.onnx'
if not best.is_file():
    raise FileNotFoundError(
        f'No best checkpoint found: {best}. Finish at least one training epoch first.'
    )
if not onnx.is_file() or best.stat().st_mtime_ns > onnx.stat().st_mtime_ns:
    print('Exporting current best validation checkpoint to ONNX...')
    subprocess.run(
        [sys.executable, '-m', 'resistor_model.export_segmentation',
         '--checkpoint', str(best), '--output', str(onnx)],
        cwd=str(repo), check=True,
    )
print(f'Testing best model: {onnx}')
runpy.run_path(str(repo / 'colab' / 'test_resistor_segmentation_alignment_colab.py'), run_name='__main__')
